In [1]:
# ============================================
# 📦 Cell 1: Environment Setup & Imports (Fixed)
# ============================================

import sys
import subprocess

def install_if_missing(package):
    """Installs a package in the current Jupyter environment if not present."""
    try:
        __import__(package)
    except ImportError:
        print(f"Installing {package} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

# Core dependencies
for pkg in ["torch", "torchvision", "diffusers", "transformers", 
            "accelerate", "safetensors", "torchmetrics", "lpips"]:
    install_if_missing(pkg)

# ---- Imports ----
import torch
import time
import os
from pathlib import Path
from diffusers import StableDiffusionXLPipeline
from PIL import Image
import numpy as np
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.psnr import PeakSignalNoiseRatio
from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity

# ---- Device Configuration ----
if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print(f"✅ Using device: {device}")

# ---- Paths ----
BASE_DIR = Path("./evaluation_outputs")
BASE_DIR.mkdir(exist_ok=True)
original_image_path = BASE_DIR / "original_sdxl.png"
quantized_image_path = BASE_DIR / "quantized_sdxl.png"

print("✅ Environment setup complete.")


/opt/homebrew/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Using device: mps
✅ Environment setup complete.


In [3]:
# ============================================
# 🎨 Cell 2: Load Models & Generate Images
# ============================================

from diffusers import StableDiffusionXLPipeline
import torch
from PIL import Image
import numpy as np
import random

# ---- Reproducibility ----
seed = 1234
generator = torch.manual_seed(seed)

# ---- Paths to models ----
# Base model from Stability AI
BASE_MODEL = "stabilityai/stable-diffusion-xl-base-1.0"

# Path to your quantized model directory (adjust if needed)
QUANTIZED_MODEL_PATH = "./quantized_models/sdxl_lightning_2step_mps_fp16"

# ---- Load Pipelines ----
print("Loading original SDXL model...")
pipe_base = StableDiffusionXLPipeline.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16 if device != "cpu" else torch.float32,
    variant="fp16" if device != "cpu" else None
).to(device)

print("Loading quantized model...")
pipe_quant = StableDiffusionXLPipeline.from_pretrained(
    QUANTIZED_MODEL_PATH,
    torch_dtype=torch.float16 if device != "cpu" else torch.float32
).to(device)

# ---- Prompt for evaluation ----
prompt = "A serene mountain landscape at sunset with a crystal clear lake"

# ---- Image generation parameters ----
steps = 4   # match your Lightning variant
height, width = 768, 768

# ---- Generate images ----
print("\nGenerating baseline (original) image...")
with torch.inference_mode():
    image_base = pipe_base(
        prompt,
        num_inference_steps=steps,
        guidance_scale=0,
        generator=generator,
        height=height,
        width=width
    ).images[0]

print("Generating quantized image...")
with torch.inference_mode():
    image_quant = pipe_quant(
        prompt,
        num_inference_steps=steps,
        guidance_scale=0,
        generator=generator,
        height=height,
        width=width
    ).images[0]

# ---- Save both ----
image_base.save(original_image_path)
image_quant.save(quantized_image_path)

print(f"✅ Saved baseline image:   {original_image_path}")
print(f"✅ Saved quantized image:  {quantized_image_path}")


Loading original SDXL model...


Loading pipeline components...: 100%|██████████| 7/7 [00:00<00:00, 12.10it/s]


Loading quantized model...


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]


OSError: Error no file named pytorch_model.bin, model.safetensors, tf_model.h5, model.ckpt.index or flax_model.msgpack found in directory ./quantized_models/sdxl_lightning_2step_mps_fp16/text_encoder.

In [10]:
# ============================================
# 🎨 Cell 2: Load Local Models & Generate Images
# (FP16 baseline vs Quantized)
# ============================================

from diffusers import StableDiffusionXLPipeline
import torch
from PIL import Image
import numpy as np

# ---- Reproducibility ----
seed = 1234
generator = torch.manual_seed(seed)

# ---- Paths to local models ----
# Baseline (FP16) model
BASE_MODEL = "./quantized_models/sdxl_lightning_2step_mps_fp16"

# Quantized (INT8) model path
QUANTIZED_MODEL_PATH = "./quantized_models/sdxl_lightning_2step_int8"

# ---- Load Pipelines ----
print("Loading baseline (FP16) Lightning model...")
pipe_base = StableDiffusionXLPipeline.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16 if device != "cpu" else torch.float32
).to(device)

print("Loading quantized (INT8) Lightning model...")
pipe_quant = StableDiffusionXLPipeline.from_pretrained(
    QUANTIZED_MODEL_PATH,
    torch_dtype=torch.float16 if device != "cpu" else torch.float32
).to(device)

# Enable memory-efficient features for both
pipe_base.enable_attention_slicing()
pipe_base.enable_vae_slicing()
pipe_quant.enable_attention_slicing()
pipe_quant.enable_vae_slicing()

# ---- Prompt for evaluation ----
prompt = "A serene mountain landscape at sunset with a crystal clear lake"

# ---- Generation parameters ----
steps = 2     # match your model variant (2-step)
height, width = 768, 768

# ---- Generate images ----
print("\nGenerating baseline FP16 image...")
with torch.inference_mode():
    image_base = pipe_base(
        prompt,
        num_inference_steps=steps,
        guidance_scale=0,
        generator=generator,
        height=height,
        width=width
    ).images[0]

print("Generating quantized INT8 image...")
with torch.inference_mode():
    image_quant = pipe_quant(
        prompt,
        num_inference_steps=steps,
        guidance_scale=0,
        generator=generator,
        height=height,
        width=width
    ).images[0]

# ---- Save both ----
image_base.save(original_image_path)
image_quant.save(quantized_image_path)

print(f"✅ Saved baseline image:   {original_image_path}")
print(f"✅ Saved quantized image:  {quantized_image_path}")


Loading baseline (FP16) Lightning model...


Loading pipeline components...: 100%|██████████| 7/7 [00:01<00:00,  5.08it/s]


Loading quantized (INT8) Lightning model...


ValueError: The provided pretrained_model_name_or_path "./quantized_models/sdxl_lightning_2step_int8" is neither a valid local path nor a valid repo id. Please check the parameter.